<a href="https://colab.research.google.com/github/IAmSipp/diabetes_prediction_with_NHANSES_dataset/blob/model%2Fmodel_v3/NextDecade_Diabetes_Wearable_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
!pip install pyreadstat requests

In [60]:
!pip install shap

In [81]:
import pandas as pd
import numpy as np
import pyreadstat
import requests
from io import BytesIO
import shap
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login


# **Data**

In [62]:
NHANSES_URLS = [
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt',
    # 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/SLQ_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TCHOL_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/PAQ_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.xpt'
    ]

In [63]:
def download_nhanes_with_labels(url):
    response = requests.get(url)
    df, meta = pyreadstat.read_xport(BytesIO(response.content), encoding='latin1')
    return df, meta

def import_nhanes_xpt(urls):
    if isinstance(urls, str): urls = [urls]

    final_df = pd.DataFrame()
    all_labels = {}

    for i, url in enumerate(urls):
        df, meta = download_nhanes_with_labels(url)

        all_labels.update(meta.column_names_to_labels)

        if i == 0:
            final_df = df
        else:
            final_df = pd.merge(final_df, df, on='SEQN', how='inner')

    all_labels['SEQN'] = 'Patient_ID'
    final_df = final_df.rename(columns=all_labels)

    return final_df

In [64]:
raw_df = import_nhanes_xpt(NHANSES_URLS)
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6337 entries, 0 to 6336
Data columns (total 56 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Patient_ID                                6337 non-null   float64
 1   Data release cycle                        6337 non-null   float64
 2   Interview/Examination status              6337 non-null   float64
 3   Gender                                    6337 non-null   float64
 4   Age in years at screening                 6337 non-null   float64
 5   Age in months at screening - 0 to 24 mos  0 non-null      float64
 6   Race/Hispanic origin                      6337 non-null   float64
 7   Race/Hispanic origin w/ NH Asian          6337 non-null   float64
 8   Six-month time period                     6337 non-null   float64
 9   Age in months at exam - 0 to 19 years     264 non-null    float64
 10  Served active duty in US Armed Force

# **Data Reduction & Data Smapling**

In [65]:
selected_features = {
    # ข้อมูลพื้นฐาน
    'Patient_ID': 'ID',
    'Gender': 'Gender',
    'Age in years at screening': 'Age',

    # 'BMI Category - Children/Youth': 'BMI',

    # การนอน
    'Sleep hours - weekdays or workdays': 'Sleep_Hours',

    # กิจกรรม (นาทีต่อวัน)
    'Minutes sedentary activity': 'Sedentary_Minutes',
    'Minutes moderate LTPA': 'Moderate_Activity_Minutes',
    'Minutes vigorous LTPA': 'Vigorous_Activity_Minutes',

    # สุขภาพอื่นๆ (เอาไว้เสริมความแม่นยำ)
    'Total Cholesterol (mg/dL)': 'Cholesterol',

    # หัวใจและความดัน (เก็บมาทุกรอบก่อนเพื่อทำ Feature Engineering)
    'Pulse - 1st oscillometric reading': 'Pulse_1',
    'Pulse - 2nd oscillometric reading': 'Pulse_2',
    'Pulse - 3rd oscillometric reading': 'Pulse_3',
    'Systolic - 1st oscillometric reading': 'Sys_1',
    'Systolic - 2nd oscillometric reading': 'Sys_2',
    'Systolic - 3rd oscillometric reading': 'Sys_3',
    'Diastolic - 1st oscillometric reading': 'Dia_1',
    'Diastolic - 2nd oscillometric reading': 'Dia_2',
    'Diastolic - 3rd oscillometric reading': 'Dia_3',

    # ตัวแปรเป้าหมาย (Label)
    'Glycohemoglobin (%)': 'HbA1c'
}

In [66]:
selected_df = raw_df[selected_features.keys()].rename(columns=selected_features)

In [67]:
selected_df = selected_df.dropna(subset=['HbA1c'])

selected_df['Resting_HR'] = selected_df[['Pulse_2', 'Pulse_3']].mean(axis=1)
selected_df['Systolic_BP'] = selected_df[['Sys_2', 'Sys_3']].mean(axis=1)
selected_df['Diastolic_BP'] = selected_df[['Dia_2', 'Dia_3']].mean(axis=1)

# selected_df['Is_Diabetes'] = (selected_df['HbA1c'] >= 6.5).astype(int)

cols_to_drop = ['ID', 'Pulse_1', 'Pulse_2', 'Pulse_3', 'Sys_1', 'Sys_2', 'Sys_3', 'Dia_1', 'Dia_2', 'Dia_3']
selected_first_clean_df = selected_df.drop(columns=cols_to_drop)

In [68]:
selected_first_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6002 entries, 0 to 6336
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Gender                     6002 non-null   float64
 1   Age                        6002 non-null   float64
 2   Sleep_Hours                5939 non-null   float64
 3   Sedentary_Minutes          5996 non-null   float64
 4   Moderate_Activity_Minutes  4781 non-null   float64
 5   Vigorous_Activity_Minutes  2725 non-null   float64
 6   Cholesterol                5714 non-null   float64
 7   HbA1c                      6002 non-null   float64
 8   Resting_HR                 5807 non-null   float64
 9   Systolic_BP                5807 non-null   float64
 10  Diastolic_BP               5807 non-null   float64
dtypes: float64(11)
memory usage: 562.7 KB


In [69]:
selected_first_clean_df

,Gender,Age,Sleep_Hours,Sedentary_Minutes,Moderate_Activity_Minutes,Vigorous_Activity_Minutes,Cholesterol,HbA1c,Resting_HR,Systolic_BP,Diastolic_BP
0,1.0,43.0,9.5,360.0,45.0,45.0,264.0,5.6,80.5,131.5,95.0
1,1.0,66.0,9.0,480.0,45.0,45.0,214.0,5.6,72.0,115.0,76.0
2,2.0,44.0,8.0,240.0,20.0,NaN,187.0,6.2,80.0,108.0,78.0
3,1.0,34.0,7.5,180.0,30.0,30.0,183.0,5.1,64.0,117.5,74.5
4,2.0,68.0,3.0,1200.0,NaN,NaN,203.0,5.9,78.5,140.5,76.0
...,...,...,...,...,...,...,...,...,...,...,...
6331,2.0,69.0,8.0,360.0,60.0,NaN,110.0,8.1,75.0,126.0,68.5
6332,2.0,76.0,9.0,480.0,40.0,NaN,180.0,6.0,70.5,146.0,78.5
6333,2.0,49.0,7.0,480.0,15.0,NaN,205.0,6.2,68.5,131.5,72.5
6335,1.0,40.0,8.0,240.0,15.0,NaN,255.0,5.2,81.0,126.5,81.5


In [74]:
initial_count = len(selected_first_clean_df)

# กฎที่ 1: อายุ
condition_age = (selected_first_clean_df['Age'] >= 0) & (selected_first_clean_df['Age'] <= 100)
# กฎที่ 2: ชีพจร
condition_hr = (selected_first_clean_df['Resting_HR'] >= 30) & (selected_first_clean_df['Resting_HR'] <= 200)
# กฎที่ 3: ชั่วโมงการนอน
condition_sleep = (selected_first_clean_df['Sleep_Hours'] >= 2) & (selected_first_clean_df['Sleep_Hours'] <= 16)
# กฎที่ 4: ความดันโลหิต
condition_bp = (selected_first_clean_df['Systolic_BP'] > selected_first_clean_df['Diastolic_BP']) & \
               (selected_first_clean_df['Systolic_BP'] <= 250) & \
               (selected_first_clean_df['Diastolic_BP'] >= 40)
# กฎที่ 5: กิจกรรม
condition_activity = (selected_first_clean_df['Sedentary_Minutes'] <= 1440) & \
                     (selected_first_clean_df['Moderate_Activity_Minutes'] <= 1440)

all_conditions = condition_age & condition_hr & condition_sleep & condition_bp & condition_activity

# กรองเอาเฉพาะข้อมูลที่ถูกกฎ (หรือเป็น NaN ไว้ไปจัดการทีหลัง)
cleaned_df = selected_first_clean_df[all_conditions | selected_first_clean_df.isna().any(axis=1)].copy()

final_count = len(cleaned_df)
print(f"ลบข้อมูลที่ผิดปกติออกไป: {initial_count - final_count} แถว")
print(f"คงเหลือข้อมูลคุณภาพ: {final_count} แถว\n")

ลบข้อมูลที่ผิดปกติออกไป: 8 แถว
คงเหลือข้อมูลคุณภาพ: 5994 แถว



In [77]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5994 entries, 0 to 6336
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Gender                     5994 non-null   float64
 1   Age                        5994 non-null   float64
 2   Sleep_Hours                5931 non-null   float64
 3   Sedentary_Minutes          5988 non-null   float64
 4   Moderate_Activity_Minutes  4773 non-null   float64
 5   Vigorous_Activity_Minutes  2717 non-null   float64
 6   Cholesterol                5706 non-null   float64
 7   HbA1c                      5994 non-null   float64
 8   Resting_HR                 5799 non-null   float64
 9   Systolic_BP                5799 non-null   float64
 10  Diastolic_BP               5799 non-null   float64
dtypes: float64(11)
memory usage: 561.9 KB


# **Feature Engineering**

In [76]:
def create_aggregated_features(df):
    """สร้างข้อมูลพฤติกรรมจำลองแบบสรุปภาพรวม สำหรับ 1 คนไข้"""
    df_agg = df.copy()

    # คำนวณเป็นค่าเฉลี่ย โดยถ้าไม่มีข้อมูล ให้เติมค่ากลางลงไปชั่วคราว
    # (หมายเหตุ: ในแอปจริง ค่าพวกนี้จะได้จากการ Average ข้อมูล 7 วันของ User)
    df_agg['Avg_Daily_Steps'] = df_agg['Moderate_Activity_Minutes'].fillna(0) * 80 + np.random.normal(3000, 1000, len(df_agg))
    df_agg['Avg_Daily_Steps'] = df_agg['Avg_Daily_Steps'].clip(lower=1000).astype(int)

    df_agg['Avg_Sleep_Hours'] = df_agg['Sleep_Hours'].fillna(7.0) + np.random.normal(0, 0.5, len(df_agg))
    df_agg['Avg_Sleep_Hours'] = df_agg['Avg_Sleep_Hours'].clip(lower=4, upper=12).round(1)

    df_agg['Avg_Resting_HR'] = df_agg['Resting_HR'].fillna(72)

    # สร้างโน้ตข้อความจำลองจากแพทย์
    def generate_clinical_notes(row):
        notes = []
        if pd.notna(row['Systolic_BP']) and row['Systolic_BP'] > 140:
            notes.append("Patient has elevated blood pressure.")
        # if pd.notna(row['BMI']) and row['BMI'] > 30:
        #     notes.append("Patient is classified as obese.")
        if pd.notna(row['HbA1c']) and row['HbA1c'] >= 6.5:
             notes.append("Patient reports frequent thirst and urination.")

        if not notes:
             return np.nan
        return " ".join(notes)

    df_agg['Clinical_Notes'] = df_agg.apply(generate_clinical_notes, axis=1)
    return df_agg

# เรียกใช้ฟังก์ชันโดยใส่ cleaned_df เข้าไป
final_feature_df = create_aggregated_features(cleaned_df)
print(f"เตรียมข้อมูล Aggregated เสร็จสิ้น Shape: {final_feature_df.shape}\n")

เตรียมข้อมูล Aggregated เสร็จสิ้น Shape: (5994, 15)



In [78]:
def calculate_risk_score(hba1c):
    if pd.isna(hba1c):
        return np.nan
    if hba1c <= 5.6:
        score = 1 + ((max(hba1c, 4.0) - 4.0) / (5.6 - 4.0)) * 32
    elif hba1c <= 6.4:
        score = 34 + ((hba1c - 5.7) / (6.4 - 5.7)) * 32
    else:
        capped = min(hba1c, 10.0)
        score = 67 + ((capped - 6.5) / (10.0 - 6.5)) * 33
    return int(np.clip(score, 1, 100))

final_feature_df['Risk_Score'] = final_feature_df['HbA1c'].apply(calculate_risk_score)

In [79]:
final_feature_df = final_feature_df.dropna(subset=['Risk_Score'])
print(f"แปลงคะแนน 1-100 เสร็จสิ้น คงเหลือข้อมูลสำหรับเทรน: {len(final_feature_df)} แถว\n")

แปลงคะแนน 1-100 เสร็จสิ้น คงเหลือข้อมูลสำหรับเทรน: 5994 แถว



In [83]:
def create_gemma_prompt(row):
    instruction = (
        "You are an expert medical AI. Analyze the patient's aggregated health and wearable data. "
        "Predict their diabetes risk score on a scale from 1 to 100 (1=Lowest risk, 100=Highest risk). "
        "Return ONLY a valid JSON object with the key 'risk_score' and an integer value. Do not include any explanations."
    )

    features = []
    if pd.notna(row['Age']): features.append(f"- Age: {row['Age']} years")
    if pd.notna(row['Gender']):
        gender_str = "Male" if row['Gender'] == 1 else "Female"
        features.append(f"- Gender: {gender_str}")
    if pd.notna(row['Avg_Resting_HR']): features.append(f"- Avg Resting HR: {row['Avg_Resting_HR']} bpm")
    if pd.notna(row['Avg_Daily_Steps']): features.append(f"- Avg Daily Steps: {row['Avg_Daily_Steps']}")
    if pd.notna(row['Avg_Sleep_Hours']): features.append(f"- Avg Sleep: {row['Avg_Sleep_Hours']} hours")

    context = "\n".join(features)
    target_json = json.dumps({"risk_score": int(row['Risk_Score'])})

    # โครงสร้าง Prompt ของ Gemma (จุดนี้ที่ต้องเปลี่ยน)
    prompt = (
        f"<start_of_turn>user\n{instruction}\n\nPatient Data:\n{context}<end_of_turn>\n"
        f"<start_of_turn>model\n```json\n{target_json}\n```<end_of_turn>"
    )
    return prompt

In [84]:
final_feature_df['LLM_Prompt'] = final_feature_df.apply(create_gemma_prompt, axis=1)
print("=== ตัวอย่างข้อมูล Prompt ที่พร้อมส่งให้ SFTTrainer ===")
print(final_feature_df['LLM_Prompt'].iloc[0])

=== ตัวอย่างข้อมูล Prompt ที่พร้อมส่งให้ SFTTrainer ===
<start_of_turn>user
You are an expert medical AI. Analyze the patient's aggregated health and wearable data. Predict their diabetes risk score on a scale from 1 to 100 (1=Lowest risk, 100=Highest risk). Return ONLY a valid JSON object with the key 'risk_score' and an integer value. Do not include any explanations.

Patient Data:
- Age: 43.0 years
- Gender: Male
- Avg Resting HR: 80.5 bpm
- Avg Daily Steps: 6778
- Avg Sleep: 9.9 hours<end_of_turn>
<start_of_turn>model
```json
{"risk_score": 33}
```<end_of_turn>


In [85]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. โหลดโมเดล MedGemma
model_id = "google/medgemma-1.5-4b-it"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.config.use_cache = False

# 3. โหลด Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. ตั้งค่า LoRA (แค่ค่า config ไม่ต้องรัน get_peft_model เอง)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

# 5. ตั้งค่า Training Arguments (SFTConfig)
training_args = SFTConfig(
    output_dir="./results",
    dataset_text_field="LLM_Prompt",
    max_seq_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,      # สำหรับ T4 GPU
    bf16=False,
)

# 6. สร้าง SFTTrainer (ส่ง peft_config เข้าไปให้ Trainer จัดการให้)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer, # ใช้แทน tokenizer ในเวอร์ชันใหม่
    args=training_args,
)

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
trainer.train()